In [0]:
%sql
SHOW CATALOGS;

catalog
bootcamp_catalog
samples
system
workspace


In [0]:
import json

products_path = "/Workspace/Users/tehreemrimsha4@gmail.com/retail-raw-cone-project/products.json"
exchange_rates_path = "/Workspace/Users/tehreemrimsha4@gmail.com/retail-raw-cone-project/exchange_rates.json"

with open(products_path, "r", encoding="utf-8") as f:
    products_data = json.load(f)

with open(exchange_rates_path, "r", encoding="utf-8") as f:
    exchange_rate_data = json.load(f)

print("Products records:", len(products_data))
print("Exchange records:", len(exchange_rate_data))
print("First product:", products_data[0] if products_data else "No data")
print("First exchange rate:", exchange_rate_data[0] if exchange_rate_data else "No data")

Products records: 20
Exchange records: 3
First product: {'product_id': 1, 'product_title': 'Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops', 'price': 109.95, 'product_category': "men's clothing", 'description': 'Your perfect pack for everyday use and walks in the forest. Stash your laptop (up to 15 inches) in the padded sleeve, your everyday', 'rating_rate': 3.9, 'rating_count': 120}
First exchange rate: {'date': '2026-04-20', 'base_currency': 'EUR', 'target_currency': 'USD', 'exchange_rate': 1.176}


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_title", StringType(), True),
    StructField("product_category", StringType(), True),
    StructField("description", StringType(), True),
    StructField("rating_rate", DoubleType(), True),  
    StructField("rating_count", LongType(), True),
])

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

exchange_rate_schema = StructType([
    StructField("date", StringType(), True),
    StructField("base_currency", StringType(), True),
    StructField("target_currency", StringType(), True),
    StructField("exchange_rate", DoubleType(), True),
])

In [0]:
products_df = spark.createDataFrame(products_data, schema = products_schema)
exchange_rate_df = spark.createDataFrame(exchange_rate_data, schema = exchange_rate_schema)

In [0]:
from pyspark.sql.functions import trim, col, coalesce, lit

products_clean = (
    products_df
    .withColumn("product_title", trim(col("product_title")))
    .withColumn("product_category", trim(col("product_category")))
    .withColumn("description", trim(col("description")))
    .withColumn("rating_rate", coalesce(col("rating_rate"), lit(0.0)))
    .withColumn("rating_count", coalesce(col("rating_count"), lit(0)))
    .dropDuplicates(["product_id"])
)

display(products_clean)
products_clean.printSchema()

product_id,product_title,product_category,description,rating_rate,rating_count
1,"Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops",men's clothing,"Your perfect pack for everyday use and walks in the forest. Stash your laptop (up to 15 inches) in the padded sleeve, your everyday",3.9,120
10,SanDisk SSD PLUS 1TB Internal SSD - SATA III 6 Gb/s,electronics,"Easy upgrade for faster boot up, shutdown, application load and response (As compared to 5400 RPM SATA 2.5” hard drive; Based on published specifications and internal benchmarking tests using PCMark vantage scores) Boosts burst write performance, making it ideal for typical PC workloads The perfect balance of performance and reliability Read/write speeds of up to 535MB/s/450MB/s (Based on internal testing; Performance may vary depending upon drive capacity, host device, OS and application.)",2.9,470
11,Silicon Power 256GB SSD 3D NAND A55 SLC Cache Performance Boost SATA III 2.5,electronics,"3D NAND flash are applied to deliver high transfer speeds Remarkable transfer speeds that enable faster bootup and improved overall system performance. The advanced SLC Cache Technology allows performance boost and longer lifespan 7mm slim design suitable for Ultrabooks and Ultra-slim notebooks. Supports TRIM command, Garbage Collection technology, RAID, and ECC (Error Checking & Correction) to provide the optimized performance and enhanced reliability.",4.8,319
12,WD 4TB Gaming Drive Works with Playstation 4 Portable External Hard Drive,electronics,"Expand your PS4 gaming experience, Play anywhere Fast and easy, setup Sleek design with high capacity, 3-year manufacturer's limited warranty",4.8,400
13,Acer SB220Q bi 21.5 inches Full HD (1920 x 1080) IPS Ultra-Thin,electronics,21. 5 inches Full HD (1920 x 1080) widescreen IPS display And Radeon free Sync technology. No compatibility for VESA Mount Refresh Rate: 75Hz - Using HDMI port Zero-frame design | ultra-thin | 4ms response time | IPS panel Aspect ratio - 16: 9. Color Supported - 16. 7 million colors. Brightness - 250 nit Tilt angle -5 degree to 15 degree. Horizontal viewing angle-178 degree. Vertical viewing angle-178 degree 75 hertz,2.9,250
14,Samsung 49-Inch CHG90 144Hz Curved Gaming Monitor (LC49HG90DMNXZA) – Super Ultrawide Screen QLED,electronics,"49 INCH SUPER ULTRAWIDE 32:9 CURVED GAMING MONITOR with dual 27 inch screen side by side QUANTUM DOT (QLED) TECHNOLOGY, HDR support and factory calibration provides stunningly realistic and accurate color and contrast 144HZ HIGH REFRESH RATE and 1ms ultra fast response time work to eliminate motion blur, ghosting, and reduce input lag",2.2,140
15,BIYLACLESEN Women's 3-in-1 Snowboard Jacket Winter Coats,women's clothing,"Note:The Jackets is US standard size, Please choose size as your usual wear Material: 100% Polyester; Detachable Liner Fabric: Warm Fleece. Detachable Functional Liner: Skin Friendly, Lightweigt and Warm.Stand Collar Liner jacket, keep you warm in cold weather. Zippered Pockets: 2 Zippered Hand Pockets, 2 Zippered Pockets on Chest (enough to keep cards or keys)and 1 Hidden Pocket Inside.Zippered Hand Pockets and Hidden Pocket keep your things secure. Humanized Design: Adjustable and Detachable Hood and Adjustable cuff to prevent the wind and water,for a comfortable fit. 3 in 1 Detachable Design provide more convenience, you can separate the coat and inner as needed, or wear it together. It is suitable for different season and help you adapt to different climates",2.6,235
16,Lock and Love Women's Removable Hooded Faux Leather Moto Biker Jacket,women's clothing,"100% POLYURETHANE(shell) 100% POLYESTER(lining) 75% POLYESTER 25% COTTON (SWEATER), Faux leather material for style and comfort / 2 pockets of front, 2-For-One Hooded denim style faux leather jacket, Button detail on waist / Detail stitching at sides, HAND WASH ONLY / DO NOT BLEACH / LINE DRY / DO NOT IRON",2.9,340
17,Rain Jacket Women Windbreaker Striped Climbing Raincoats,women's clothing,"Lightweight perfet 

root
 |-- product_id: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- description: string (nullable = true)
 |-- rating_rate: double (nullable = false)
 |-- rating_count: long (nullable = false)



In [0]:
from pyspark.sql.functions import trim, upper, to_date, round

exchange_rate_clean = (
    exchange_rate_df
    .withColumn("base_currency", upper(trim(col("base_currency"))))
    .withColumn("target_currency", upper(trim(col("target_currency"))))
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
    .withColumn("exchange_rate", round(col("exchange_rate"), 2))
    .dropDuplicates()
)

display(exchange_rate_clean)
exchange_rate_clean.printSchema()

date,base_currency,target_currency,exchange_rate
2026-04-20,EUR,USD,1.18
2026-04-20,GBP,USD,1.18
2026-04-20,JPY,USD,1.18


root
 |-- date: date (nullable = true)
 |-- base_currency: string (nullable = true)
 |-- target_currency: string (nullable = true)
 |-- exchange_rate: double (nullable = true)



In [0]:
spark.sql("use catalog bootcamp_catalog")
spark.sql("create schema if not exists silver")
spark.sql("use schema silver")

DataFrame[]

In [0]:
(products_clean.write.format("delta").mode("overwrite").saveAsTable("bootcamp_catalog.silver.dim_products"))

In [0]:
(exchange_rate_clean.write.format("delta").mode("overwrite").saveAsTable("bootcamp_catalog.silver.dim_exchange_rates"))

In [0]:
display(spark.sql("select * from bootcamp_catalog.silver.dim_products"))
display(spark.sql("select * from bootcamp_catalog.silver.dim_exchange_rates"))

product_id,product_title,product_category,description,rating_rate,rating_count
2,Mens Casual Premium Slim Fit T-Shirts,men's clothing,"Slim-fitting style, contrast raglan long sleeve, three-button henley placket, light weight & soft fabric for breathable and comfortable wearing. And Solid stitched shirts with round neck made for durability and a great fit for casual fashion wear and diehard baseball fans. The Henley style round neckline includes a three-button placket.",4.1,259
1,"Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops",men's clothing,"Your perfect pack for everyday use and walks in the forest. Stash your laptop (up to 15 inches) in the padded sleeve, your everyday",3.9,120
3,Mens Cotton Jacket,men's clothing,"great outerwear jackets for Spring/Autumn/Winter, suitable for many occasions, such as working, hiking, camping, mountain/rock climbing, cycling, traveling or other outdoors. Good gift choice for you or your family member. A warm hearted love to Father, husband or son in this thanksgiving or Christmas Day.",4.7,500
4,Mens Casual Slim Fit,men's clothing,"The color could be slightly different between on the screen and in practice. / Please note that body builds vary by person, therefore, detailed size information should be reviewed below on the product description.",2.1,430
5,John Hardy Women's Legends Naga Gold & Silver Dragon Station Chain Bracelet,jewelery,"From our Legends Collection, the Naga was inspired by the mythical water dragon that protects the ocean's pearl. Wear facing inward to be bestowed with love and abundance, or outward for protection.",4.6,400
6,Solid Gold Petite Micropave,jewelery,Satisfaction Guaranteed. Return or exchange any order within 30 days.Designed and sold by Hafeez Center in the United States. Satisfaction Guaranteed. Return or exchange any order within 30 days.,3.9,70
7,White Gold Plated Princess,jewelery,"Classic Created Wedding Engagement Solitaire Diamond Promise Ring for Her. Gifts to spoil your love more for Engagement, Wedding, Anniversary, Valentine's Day...",3.0,400
8,Pierced Owl Rose Gold Plated Stainless Steel Double,jewelery,Rose Gold Plated Double Flared Tunnel Plug Earrings. Made of 316L Stainless Steel,1.9,100
9,WD 2TB Elements Portable External Hard Drive - USB 3.0,electronics,"USB 3.0 and USB 2.0 Compatibility Fast data transfers Improve PC Performance High Capacity; Compatibility Formatted NTFS for Windows 10, Windows 8.1, Windows 7; Reformatting may be required for other operating systems; Compatibility may vary depending on user’s hardware configuration and operating system",3.3,203
10,SanDisk SSD PLUS 1TB Internal SSD - SATA III 6 Gb/s,electronics,"Easy upgrade for faster boot up, shutdown, application load and response (As compared to 5400 RPM SATA 2.5” hard drive; Based on published specifications and internal benchmarking tests using PCMark vantage scores) Boosts burst write performance, making it ideal for typical PC workloads The perfect balance of performance and reliability Read/write speeds of up to 535MB/s/450MB/s (Based on internal testing; Performance may vary depending upon drive capacity, host device, OS and application.)",2.9,470


date,base_currency,target_currency,exchange_rate
2026-04-20,EUR,USD,1.18
2026-04-20,GBP,USD,1.18
2026-04-20,JPY,USD,1.18
